# Charts

Seven short lessons, each on one real dataset from [The Pudding](https://pudding.cool), taken
straight from [their public data repo](https://github.com/the-pudding/data).

1. A bar, for comparing two things
2. Bars hide the spread
3. A line, for change over time
4. Divide by whatever grew anyway
5. When the colour is the data
6. The category is a choice
7. The title is the finding

Then five harder ones, built out of the same parts:

8. Small multiples
9. A bump chart
10. A dumbbell chart
11. A heatmap
12. A scatter that says no

No new library. `pandas` to hold the numbers, `matplotlib` to draw them.

In [ ]:
import io
import pandas as pd
import matplotlib.pyplot as plt
import requests

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "font.size": 10})
RUST, BLUE, GREY = "#A34526", "#2E6E8E", "#BBB5AE"

PUDDING = "https://raw.githubusercontent.com/the-pudding/data/master/"

def pudding(path):
    """Read one of The Pudding's CSVs straight off GitHub."""
    return pd.read_csv(io.StringIO(requests.get(PUDDING + path, timeout=60).text))

## 1 · A bar, for comparing two things

*Women's Pockets are Inferior* (2018). Someone measured the pockets in 80 pairs of jeans.

In [ ]:
jeans = pudding("pockets/measurements.csv")
print(len(jeans), "pairs,", jeans["brand"].nunique(), "brands")
jeans[["brand", "style", "menWomen", "maxHeightFront", "maxWidthFront"]].head()

In [ ]:
depth = jeans.groupby("menWomen")["maxHeightFront"].mean()
print(depth.round(1))

fig, ax = plt.subplots(figsize=(4.5, 3.2))
ax.bar(["men's", "women's"], [depth["men"], depth["women"]], color=[BLUE, RUST], width=0.55)
for i, v in enumerate([depth["men"], depth["women"]]):
    ax.text(i, v + 0.4, f"{v:.1f}", ha="center")
ax.set_ylabel("front pocket depth (cm)")
ax.set_title("Women's jeans have shallower pockets")
plt.tight_layout(); plt.show()

Two numbers, two bars. The bars start at zero, so twice as tall means twice as deep.

## 2 · Bars hide the spread

Those two bars are 80 pairs of jeans squashed into two numbers. Draw all 80.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for i, (group, colour) in enumerate([("men", BLUE), ("women", RUST)]):
    vals = jeans.loc[jeans["menWomen"] == group, "maxHeightFront"]
    ax.scatter(vals, [i] * len(vals), color=colour, alpha=0.6, s=45)
    ax.scatter(vals.mean(), i, color="black", marker="|", s=400)
ax.set_yticks([0, 1], ["men's", "women's"])
ax.set_ylim(-0.7, 1.7)
ax.set_xlabel("front pocket depth (cm)")
ax.set_title("Every pair, with the average marked")
plt.tight_layout(); plt.show()

shallowest_men = jeans.loc[jeans["menWomen"] == "men", "maxHeightFront"].min()
deeper = (jeans.loc[jeans["menWomen"] == "women", "maxHeightFront"] > shallowest_men).sum()
print(f"{deeper} of 40 women's pairs beat the shallowest men's pair ({shallowest_men} cm)")

The two groups barely touch. One pair out of forty. The bar chart was true but the dots are the
argument.

## 3 · A line, for change over time

*The Names in Songs* — every first name sung on the Billboard Hot 100, 1958 to 2019.

In [ ]:
songs = pudding("names-in-songs/unique.csv")
songs["year"] = pd.to_numeric(songs["year"], errors="coerce")
people = songs[songs["person"] == True]
people = people[people["year"] < 2019]          # 2019 is a part-year, it would dip for no reason
print(len(people), "name mentions,", int(people["year"].min()), "to", int(people["year"].max()))
people[["artist", "song", "name", "year"]].head()

In [ ]:
per_year = people.groupby("year").size()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(per_year.index, per_year.values, color=RUST, lw=2)
ax.set_ylabel("names sung")
ax.set_title("Names in hit songs, per year")
plt.tight_layout(); plt.show()
print("1960:", per_year[1960], " 2018:", per_year[2018])

Ten times as many names as in 1960. Believe it?

(The last year in the file stops mid-year, so it is dropped. A half-year plotted next to full
ones looks like a crash.)

## 4 · Divide by whatever grew anyway

The dataset has more songs in it every year too. So of course it has more names.

In [ ]:
counted = pd.DataFrame({"names": people.groupby("year").size(),
                        "songs": people.groupby("year")["song"].nunique()})
counted["per_song"] = counted["names"] / counted["songs"]

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.2), sharex=True)
a.plot(counted.index, counted["names"], color=RUST, lw=2)
a.set_title("Names, counted")
b.plot(counted.index, counted["per_song"], color=BLUE, lw=2)
b.set_ylim(0, 2.6)
b.set_title("Names per song")
plt.tight_layout(); plt.show()

print(counted.loc[[1960, 1980, 2000, 2018]].round(2).to_string())

The left chart rises tenfold. The right one goes from about 1.5 to about 2.2. Most of the first
chart was the dataset growing, not songwriting changing.

Ask this of every count you plot: what else got bigger?

## 5 · When the colour is the data

*The Naked Truth* — 625 foundation shades from 36 makeup brands, each with its own hex code.

In [ ]:
shades = pudding("makeup-shades/shades.csv")
print(len(shades), "shades,", shades["brand"].nunique(), "brands")
shades[["brand", "product", "hex", "L"]].head()

In [ ]:
# L is lightness, 0 darkest to 100 lightest. Draw each shade in its own colour.
BRANDS = ["Fenty", "Maybelline", "MAC", "Estée Lauder", "L'Oréal"]
fig, ax = plt.subplots(figsize=(7.5, 3))
for i, brand in enumerate(BRANDS):
    rows = shades[shades["brand"] == brand].sort_values("L")
    ax.scatter(rows["L"], [i] * len(rows), c="#" + rows["hex"], s=90,
               edgecolor="#444", linewidth=0.4)
ax.set_yticks(range(len(BRANDS)), BRANDS)
ax.set_xlabel("lightness (0 darkest, 100 lightest)")
ax.set_title("Every shade, in its own colour")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

The marks are the data. No legend needed, because nothing had to be translated into a key.

Colour earns its place when it *is* the measurement. The rest of the time a second colour is
usually one more thing to decode.

## 6 · The category is a choice

Back to the songs. Which names get sung most?

In [ ]:
top = people["name"].value_counts().head(10)
print(top.to_string())

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(top.index[::-1], top.values[::-1], color=GREY)
ax.set_title("Most-sung names")
plt.tight_layout(); plt.show()

"Baby" is 23% of every name in the dataset, and it is not a name. Somebody decided it counted.

Drop it and the chart is about something else.

In [ ]:
real = top.drop(["Baby", "Jesus"])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.barh(real.index[::-1], real.values[::-1], color=RUST)
ax.set_title("Most-sung names, without Baby and Jesus")
plt.tight_layout(); plt.show()

Neither chart is wrong. They answer different questions, and the difference is a decision
somebody made before any code ran. Write that decision down.

## 7 · The title is the finding

Same numbers, three titles. Only one of them tells the reader what to see.

In [ ]:
depth = jeans.groupby("menWomen")["maxHeightFront"].mean()
TITLES = ["Figure 3", "Front pocket depth by gender",
          "Women's jean pockets are 38% shallower"]

fig, axes = plt.subplots(1, 3, figsize=(10, 2.8), sharey=True)
for ax, title in zip(axes, TITLES):
    ax.bar(["men's", "women's"], [depth["men"], depth["women"]], color=[BLUE, RUST], width=0.55)
    ax.set_title(title, fontsize=10)
plt.tight_layout(); plt.show()

print(f"{100 * (1 - depth['women'] / depth['men']):.0f}% shallower")

The third one is the only one doing any work. Write the title last, once you know what the
chart says, and make it a sentence.

# Part two: harder charts

Everything below is built from the same three moves: put marks at coordinates, repeat, label.
Nothing new gets imported.

## 8 · Small multiples

*Women in Headlines* measured how emotionally loaded the wording of news headlines is, 0 to 1,
for headlines about women against all headlines.

Four countries on one chart is eight lines and a legend. Give each country its own panel with
the same axes, and you compare by looking across.

In [ ]:
tone = pudding("women-in-headlines/polarity_comparison_country_time.csv")
MEANS = ["women_polarity_mean", "all_polarity_mean"]

# These columns never legitimately sit below 0.2, so the exact zeros are missing values
# written as 0. Left in, they draw a line plunging to the floor and back.
print("exact zeros:", int((tone[MEANS] == 0).sum().sum()))
tone[MEANS] = tone[MEANS].replace(0, float("nan"))

countries = sorted(tone["country"].unique())
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9), sharey=True, sharex=True)
for ax, country in zip(axes, countries):
    one = tone[tone["country"] == country].sort_values("year")
    ax.plot(one["year"], one["all_polarity_mean"], color=GREY, lw=2)
    ax.plot(one["year"], one["women_polarity_mean"], color=RUST, lw=2)
    ax.set_title(country)
    ax.set_xticks([2010, 2015, 2020])
axes[0].set_ylim(0.10, 0.45)                     # low enough that nothing is cut off
axes[0].set_ylabel("loaded wording")
axes[0].text(2010.4, 0.40, "about women", color=RUST, fontsize=9)
axes[0].text(2010.4, 0.135, "all headlines", color=GREY, fontsize=9)
plt.tight_layout(); plt.show()

Rust above grey in all four panels. One shape, repeated, read four times.

Three rules make small multiples work: the same axes everywhere, an order that means something,
and labels once rather than in every panel. The gaps are years with too few headlines to
average, and shared axes make them visible instead of hiding them.

Two things the panels expose that a single chart would have buried. South Africa's last point
runs off the top of the axis, off a handful of headlines: clipping it is a choice, so say so in
the caption. And the US baseline dives in 2016 and climbs back, which no story here explains.
Go and check a number like that before you build on it.

## 9 · A bump chart

Ranks over time. One line per name, moving up and down a top ten. It shows the churn: who
arrives, who drops out.

In [ ]:
decades = pudding("names-in-songs/timeless_names.csv")
decades = decades[decades["person"] == True]
table = decades.pivot_table(index="name", columns="decade", values="rank")
years = sorted(table.columns)
print(len(table), "names hold a top-10 place in at least one decade")

fig, ax = plt.subplots(figsize=(8, 5))
for name, row in table.iterrows():
    seen = [(y, row[y]) for y in years if pd.notna(row[y])]
    lasting = len(seen) >= 4
    ax.plot([p[0] for p in seen], [p[1] for p in seen],
            color=RUST if lasting else GREY, lw=2.2 if lasting else 1,
            marker="o", ms=5, alpha=1 if lasting else 0.55, zorder=3 if lasting else 2)
    ax.annotate(name, seen[-1], xytext=(7, 0), textcoords="offset points",
                va="center", fontsize=8, color="black" if lasting else "#8A837C",
                bbox=dict(facecolor="white", edgecolor="none", pad=1.2, alpha=0.8))

ax.set_xticks(years)
ax.set_yticks(range(1, 11))
ax.invert_yaxis()
ax.set_ylabel("rank")
ax.set_xlim(1955, 2022)
ax.set_title("Most-sung names, top ten by decade")
plt.tight_layout(); plt.show()

Rust for names that hold a place in four decades or more. Grey for the rest, which is most of
them.

That grey is the point. A bump chart is worth the effort when the crossing and the churn *are*
the finding. If every line ran flat and parallel, a table would have said it faster.

## 10 · A dumbbell chart

Two numbers per row, joined by a line. Here: each makeup brand's darkest and lightest
foundation, sorted by how dark the darkest one is.

In [ ]:
span = shades.groupby("brand")["L"].agg(["min", "max", "count"]).sort_values("min")
span = span[span["count"] >= 6]

fig, ax = plt.subplots(figsize=(7.5, 7))
for i, (brand, row) in enumerate(span.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], color=GREY, lw=2, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], color=["#3B2A1E", "#F0DCC6"],
               s=70, edgecolor="#555", linewidth=0.5, zorder=2)
    ax.text(row["max"] + 2, i, f"{int(row['count'])}", va="center", fontsize=8, color="#8A837C")

ax.set_yticks(range(len(span)), span.index)
ax.set_xlabel("lightness of the darkest and lightest shade")
ax.set_xlim(5, 105)
ax.invert_yaxis()
ax.set_title("How far down each brand goes")
plt.tight_layout(); plt.show()

The number on the right is how many shades that brand sells, because a long line built from six
shades is not the same offer as a long line built from fifty.

Sorting did the work here. Alphabetical by brand and this chart says nothing.

## 11 · A heatmap

Two things to compare against each other and one number in every cell. Every US birth from 1985
to 2015, by month.

The file has a trap in it. `stateBirths` is repeated on every county row of that state, so
adding it up counts each birth dozens of times. `countyBirths` is the one to sum.

In [ ]:
import calendar

births = pudding("births/allBirthData.csv")
grid = births.groupby(["Year", "Month"])["countyBirths"].sum().unstack()

# February is short. Without this the chart is mostly a picture of month lengths.
days = pd.DataFrame({m: [calendar.monthrange(y, m)[1] for y in grid.index] for m in grid.columns},
                    index=grid.index)
grid = grid / days

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(grid, aspect="auto", cmap="magma",
               extent=[0.5, 12.5, grid.index.max() + 0.5, grid.index.min() - 0.5])
ax.set_xticks(range(1, 13), [calendar.month_abbr[m] for m in range(1, 13)], fontsize=8)
ax.set_title("US births per day")
fig.colorbar(im, label="births per day", shrink=0.8)
plt.tight_layout(); plt.show()

print(grid.mean().round(0).astype(int).to_string())

A bright band through late summer, brightest in September, and the whole picture dimming after
2007. Neither shows up in a line of yearly totals.

Two warnings. A heatmap can only be read against its colour bar, so the choice of colour scale
is part of the argument: `magma` and `viridis` keep their order when printed grey or seen by a
colour-blind reader, while `jet` and `rainbow` invent bands that are not in the data. And a
heatmap shows you where to look, not how big the difference is. September runs about 11% above
January, which the colours oversell.

## 12 · A scatter that says no

*The Hype Machine* tracked basketball recruits ranked in the top 100 out of high school, and
what they went on to be worth in the NBA.

If the ranking worked, this chart would slope.

In [ ]:
players = pudding("hype/players.csv").dropna(subset=["rank", "nba_mean_wa"])
print(len(players), "recruits who made the NBA")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(players["rank"], players["nba_mean_wa"], color=BLUE, alpha=0.45, s=35)
ax.axhline(0, color=GREY, lw=1)

for _, row in players.nlargest(4, "nba_mean_wa").iterrows():
    ax.annotate(row["name"], (row["rank"], row["nba_mean_wa"]), xytext=(8, -2),
                textcoords="offset points", fontsize=9)

ax.set_xlabel("high-school recruit rank (1 is the most hyped)")
ax.set_ylabel("average wins added in the NBA")
ax.set_title(f"Recruit rank barely predicts anything (r = {players['rank'].corr(players['nba_mean_wa']):.2f})")
plt.tight_layout(); plt.show()

A cloud. The number one recruit is LeBron James, and the second-best career in the picture
belongs to number ninety-five.

Two things this chart cannot say. Everyone in it reached the NBA, so the recruits the ranking
got wrong in the other direction are missing entirely. And a flat cloud is evidence about these
players, not proof that scouting is worthless.

A chart showing nothing is still a result. Publish it.

## Your turn

The Pudding's [data repo](https://github.com/the-pudding/data) has about forty datasets. Pick
one and make two charts from it: the obvious one, and one that complicates it.

```python
df = pudding("dress-codes/banned_items.csv")     # or vogue/models.csv, births/births.csv
df.head()
```

Four things to check before you show anyone a chart:

1. Does the bar start at zero? If not, say why.
2. Have you hidden a spread inside an average?
3. Did the count go up, or did the dataset?
4. Does the title say what you found?